In [1]:
!pip install roboflow --quiet

In [2]:
import os
import shutil
import xml.etree.ElementTree as ET
import kagglehub
from roboflow import Roboflow
from ultralytics import YOLO
import numpy as np
import torch
import random
import math
from PIL import Image
import glob

/home/cam/miniforge3/envs/jupyter_dl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Helper functions**

In [3]:
random.seed(0)

def convert_voc_to_yolo(voc_dir, output_dir, phys_classes: dict[str, int]):
    os.makedirs(output_dir, exist_ok=True)

    # Iterating Over Annotation Files
    for xml_file in os.listdir(os.path.join(voc_dir, 'Annotations')):
        tree = ET.parse(os.path.join(voc_dir, 'Annotations', xml_file))
        root = tree.getroot()

        img_width = int(root.find('size/width').text)
        img_height = int(root.find('size/height').text)

        yolo_annotation = []
        for obj in root.findall('object'):
            class_id = phys_classes[obj.find('name').text]

            # only save merged classes 0 and 1
            if class_id <= 1:
              bbox = obj.find('bndbox')
              xmin, ymin, xmax, ymax = [float(bbox.find(tag).text) for tag in ['xmin', 'ymin', 'xmax', 'ymax']]
              x_center = (xmin + xmax) / 2 / img_width
              y_center = (ymin + ymax) / 2 / img_height
              width = (xmax - xmin) / img_width
              height = (ymax - ymin) / img_height
              yolo_annotation.append(f"{0} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        # saving the converted annotations to a text file.
        with open(os.path.join(output_dir, f"{root.find('filename').text.split('.')[0]}.txt"), 'w') as f:
            f.write("\n".join(yolo_annotation))

# Assuming pass is defined as either train path or test path
def move_files(path):
  annotations_dir = os.path.join(path, 'Annotations')
  os.makedirs(annotations_dir, exist_ok=True)

  for filename in os.listdir(path):
    if filename.endswith(".xml"):
      source_path = os.path.join(path, filename)
      destination_path = os.path.join(annotations_dir, filename)
      shutil.move(source_path, destination_path)


def organize_files(path):
  """
  Moves all .txt files to a 'labels' subdirectory and all .jpg files to an 'images' subdirectory within the given path.

  Args:
      path: The root directory containing the files to be organized.
  """
  labels_dir = os.path.join(path, 'labels')
  images_dir = os.path.join(path, 'images')

  os.makedirs(labels_dir, exist_ok=True)
  os.makedirs(images_dir, exist_ok=True)

  for filename in os.listdir(path):
    source_path = os.path.join(path, filename)
    if filename.endswith(".txt"):
      destination_path = os.path.join(labels_dir, filename)
      shutil.move(source_path, destination_path)
    elif filename.endswith(".jpg"):
      destination_path = os.path.join(images_dir, filename)
      shutil.move(source_path, destination_path)

def rename_files(folder_path):
  """
  Renames all files in a folder by stripping all characters after and including the first underscore,
  preserving the file extension. The function operates recursively.
  """
  for root, _, files in os.walk(folder_path):
    for file in files:
      if "_" in file:
        name, ext = os.path.splitext(file)
        new_name = name.split("_", 1)[0] + ext
        old_path = os.path.join(root, file)
        new_path = os.path.join(root, new_name)
        os.rename(old_path, new_path)

# converts image tensors to .jpg files, annotation tensors to .txt files,
# saves the images and annotations to respective folders
def save_image_and_annotation(i, split):
  img_tensor = images[i]
  img_array = img_tensor.squeeze().numpy().astype(np.uint8)
  img = Image.fromarray(img_array, mode="L")
  filename = f"{i:0{pad_length}d}.jpg"
  img_save_path = os.path.join(base_dir, split, "images", filename)
  img.save(img_save_path)
  kp = keypoints[i].numpy()
  kp_str = " ".join([f"{pt:.6f}" for pt in kp])
  annotation_filename = f"{i:0{pad_length}d}.txt"
  annot_save_path = os.path.join(base_dir, split, "annotations", annotation_filename)
  with open(annot_save_path, "w") as f:
    f.write(kp_str)

def process_dataset(dataset_path, prefix):
  for split in splits:
    for folder in subfolders:
      src_dir = os.path.join(dataset_path, split, folder)
      if not os.path.exists(src_dir):
        print(f"{src_dir} does not exist. skipping")
        continue
      dest_dir = os.path.join(master_dir, split, folder)
      for filename in os.listdir(src_dir):
        src_file = os.path.join(src_dir, filename)
        if os.path.isfile(src_file):
          new_filename = prefix + filename
          dest_file = os.path.join(dest_dir, new_filename)
          shutil.copy2(src_file, dest_file)

**Physiognomy dataset conversion**

In [4]:
# download physiognomy dataset
physiognomy_path = kagglehub.dataset_download("dhufrfarooq/physiognomy-dataset")

move_files(os.path.join(physiognomy_path, 'train'))
move_files(os.path.join(physiognomy_path, 'test'))
move_files(os.path.join(physiognomy_path, 'valid'))

phys_classes = {
  "brown_eye" : 0,
  "blu_eye": 1,
  "pointy_nose": 2,
  "rounded_nose": 3,
  "straight_eyebrows": 4,
  "rounded_eyebrows": 5
}

convert_voc_to_yolo(os.path.join(physiognomy_path, 'train'), os.path.join(physiognomy_path, 'train'), phys_classes)
convert_voc_to_yolo(os.path.join(physiognomy_path, 'test'), os.path.join(physiognomy_path, 'test'), phys_classes)
convert_voc_to_yolo(os.path.join(physiognomy_path, 'valid'), os.path.join(physiognomy_path, 'valid'), phys_classes)

organize_files(os.path.join(physiognomy_path, 'train'))
organize_files(os.path.join(physiognomy_path, 'test'))
organize_files(os.path.join(physiognomy_path, 'valid'))

rename_files(physiognomy_path)

**Facial Keypoint dataset conversion**

In [5]:
if not os.path.exists('facial_keypoints.npz'):
  !wget -O facial_keypoints.npz "https://www.dropbox.com/scl/fi/27qggijmythfjg04s24xq/facial_keypoints.npz?rlkey=h91gwodhrfuz8hrc7ux9qnq7s&dl=1"

data = np.load('facial_keypoints.npz')
images = data['images']
keypoints = data['keypoints']

images = torch.from_numpy(images).float()
keypoints = torch.from_numpy(keypoints).float()

In [6]:
random.seed(42)

base_dir = "dataset"
splits = ["train", "test"]
subfolders = ["images", "annotations"]

# split the data into 90/10 train test split
for split in splits:
  for folder in subfolders:
    os.makedirs(os.path.join(base_dir, split, folder), exist_ok=True)
    num_images = images.shape[0]
    indices = list(range(num_images))
    random.shuffle(indices)
    num_train = math.floor(0.9 * num_images)
    train_indices = indices[:num_train]
    test_indices = indices[num_train:]
    pad_length = len(str(num_images))

for i in train_indices:
  save_image_and_annotation(i, "train")
for i in test_indices:
  save_image_and_annotation(i, "test")

# takes the eye corner keypoints, converts to bounding boxes, and saves bounding
# boxes as YOLO style labels in "labels" folder (each label filename corresponds
# to the image filename its labeling)
for split in ["train", "test"]:
  ann_dir = os.path.join("dataset", split, "annotations")
  lbl_dir = os.path.join("dataset", split, "labels")
  os.makedirs(lbl_dir, exist_ok=True)
  for fname in os.listdir(ann_dir):
    if not fname.endswith(".txt"):
      continue
    fpath = os.path.join(ann_dir, fname)
    with open(fpath, "r") as f:
      content = f.read().strip()
      parts = content.split()
      if len(parts) < 30:
        continue
      kp = list(map(float, parts))
      li_x, li_y = kp[4], kp[5]
      lo_x, lo_y = kp[6], kp[7]
      ri_x, ri_y = kp[8], kp[9]
      ro_x, ro_y = kp[10], kp[11]
      l_cx = (li_x + lo_x) / 2.0
      l_cy = (li_y + lo_y) / 2.0
      l_w = abs(lo_x - li_x) * 1.2
      l_h = 0.5 * l_w
      r_cx = (ri_x + ro_x) / 2.0
      r_cy = (ri_y + ro_y) / 2.0
      r_w = abs(ro_x - ri_x)
      r_h = 0.5 * r_w * 1.2
      l_line = "0 {:.6f} {:.6f} {:.6f} {:.6f}".format(l_cx/96, l_cy/96, l_w/96, l_h/96)
      r_line = "0 {:.6f} {:.6f} {:.6f} {:.6f}".format(r_cx/96, r_cy/96, r_w/96, r_h/96)
      out_path = os.path.join(lbl_dir, fname)
      with open(out_path, "w") as f:
        f.write(l_line + "\n" + r_line)

# remove images and labels with invalid bounding boxes
for split in ["train", "test"]:
  labels_dir = os.path.join("dataset", split, "labels")
  images_dir = os.path.join("dataset", split, "images")

  for fname in os.listdir(labels_dir):
    if not fname.endswith(".txt"):
      continue

    label_path = os.path.join(labels_dir, fname)
    with open(label_path, "r") as f:
      lines = f.readlines()

    remove_file = False
    for line in lines:
      if line.strip() == "0 nan nan nan nan":
        remove_file = True
        break

    if remove_file:
      img_fname = os.path.splitext(fname)[0] + ".jpg"
      img_path = os.path.join(images_dir, img_fname)

      if os.path.exists(img_path):
        os.remove(img_path)
        os.remove(label_path)

**Eye Object Detect dataset**

In [7]:
rf = Roboflow(api_key="OmDjd17rKjY69b4nRRVP")
project = rf.workspace("duong-duc-cuong").project("eye_detect_object")
version = project.version(2)
dataset = version.download("yolov11")

eye_object_path = "eye_detect_object-2"

loading Roboflow workspace...
loading Roboflow project...


**Labeled Faces in the Wild**

In [9]:
fiw_path = os.path.join(os.getcwd(), 'fiw_data')
train_dir = os.path.join(fiw_path, 'train')
valid_dir = os.path.join(fiw_path, 'test')
os.makedirs(os.path.join(train_dir, 'images'), exist_ok=True)
os.makedirs(os.path.join(train_dir, 'labels'), exist_ok=True)
os.makedirs(os.path.join(valid_dir, 'images'), exist_ok=True)
os.makedirs(os.path.join(valid_dir, 'labels'), exist_ok=True)


images = [f for f in glob.iglob(os.path.join(fiw_path, 'images', '*.jpg'))]
labels = [f for f in glob.iglob(os.path.join(fiw_path, 'labels', '*.txt'))]
data_size = len(images)

# Define train test split
train_size = 0.9
valid_size = 0.1

train_len = int(data_size * 0.9)
valid_len = data_size - train_len
is_train = ([True] * train_len) + ([False] * valid_len)
random.shuffle(is_train)

# Move data into folders for Pytorch Data Loader
for idx, (image, label) in enumerate(zip(images, labels)):
    if os.path.exists(image):
        folder = train_dir if is_train[idx] else valid_dir
        shutil.copy(image, os.path.join(folder, 'images', image.split(os.path.sep)[-1]))
        shutil.copy(label, os.path.join(folder, 'labels', label.split(os.path.sep)[-1]))

**Combine all 3 datasets**

In [10]:
# filename prefix dict
datasets = {
  'physiognomy': {
    'path': physiognomy_path,
    'prefix': 'physiognomy_'
  },
  'eye_object': {
    'path': eye_object_path,
    'prefix': 'eye_object_'
  },
  'base': {
    'path': base_dir,
    'prefix': 'facial_keypoints_'
  },
  'fiw': {
    'path': fiw_path,
    'prefix': 'faces_in_the_wild'
  } 
}

# change this master_dir var to change dataset folder name
master_dir = './combined_dataset'
splits = ['train', 'test']
subfolders = ['images', 'labels']

# create dataset folders and subfolders
for split in splits:
  for subfolder in subfolders:
    os.makedirs(os.path.join(master_dir, split, subfolder), exist_ok=True)

# process all 3 original datasets
for dataset_name, data in datasets.items():
  process_dataset(data['path'], data['prefix'])

# write the combined dataset yaml file for YOLO training
yaml = """
train: ./train/images
val: ./test/images

nc: 1
names: ['eye']
"""

with open(f"{os.getcwd()}/combined_dataset/data.yaml", "w") as f:
  f.write(yaml)

In [12]:
model = YOLO('yolo11n.pt')
model.train(data=f'{os.getcwd()}/combined_dataset/data.yaml', epochs=10)

New https://pypi.org/project/ultralytics/8.3.91 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.78 🚀 Python-3.9.21 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3090 Ti, 24226MiB)
engine/trainer: task=detect, mode=train, model=yolo11n.pt, data=/home/cam/Development/csc487/eye_detection/combined_dataset/data.yaml, epochs=10, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train9, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=N

train: Scanning /home/cam/Development/csc487/eye_detection/combined_dataset/trai

train: WARNING ⚠️ /home/cam/Development/csc487/eye_detection/combined_dataset/train/images/faces_in_the_wildAbdullah_0001.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /home/cam/Development/csc487/eye_detection/combined_dataset/train/images/faces_in_the_wildAbdullah_Ahmad_Badawi_0001.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /home/cam/Development/csc487/eye_detection/combined_dataset/train/images/faces_in_the_wildAbdullah_Gul_0014.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /home/cam/Development/csc487/eye_detection/combined_dataset/train/images/faces_in_the_wildAbdullah_Gul_0019.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /home/cam/Development/csc487/eye_detection/combined_dataset/train/images/faces_in_the_wildAdel_Al-Jubeir_0002.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /home/cam/Development/csc487/eye_detection/combined_dataset/train/images/faces_in_the_wildAdrien_Brody_0002.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /home/cam/Development/csc48

train: New cache created: /home/cam/Development/csc487/eye_detection/combined_dataset/train/labels.cache


val: Scanning /home/cam/Development/csc487/eye_detection/combined_dataset/test/l

...

                   all       1096        828      0.497      0.594      0.417      0.285



10 epochs completed in 0.089 hours.
Optimizer stripped from runs/detect/train9/weights/last.pt, 5.5MB
Optimizer stripped from runs/detect/train9/weights/best.pt, 5.5MB

Validating runs/detect/train9/weights/best.pt...
Ultralytics 8.3.78 🚀 Python-3.9.21 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3090 Ti, 24226MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  m


                   all       1096        828      0.478      0.565      0.431      0.286
Speed: 0.1ms preprocess, 0.7ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to runs/detect/train9


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x70cbb08c4100>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [13]:
model.save('super.onnx')